# 0.01 · MiBici initial data exploration

**Phase 0 (exploration)** following the [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/using-the-template/) convention: `PHASE.NOTEBOOK-initials-description`.

Goals:
1. Populate `data/raw/mibici/` with the MiBici open data CSVs (unmodified).
2. Get a first overview: inventory, schema per file, dtypes, nulls and duplicates.

This notebook downloads into `data/raw`; it does not write to `data/interim` or `data/processed`. The download logic lives in `team2_mlops2026.data.raw_data_extraction`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import pandas as pd

from team2_mlops2026.data.raw_data_extraction import MANIFEST_NAME, RAW_MIBICI_DIR, ingest

pd.set_option("display.max_columns", 50)

## Parameters

The full history (2014 to date) weighs several GB. `SINCE` limits the download to months from `YYYY-MM` on; use `None` to download everything. The download is idempotent: re-running only fetches what is missing.

In [ ]:
SINCE = "2025-01"  # None = full history
WORKERS = 4  # parallel downloads
FORCE = False  # True = re-download even if it exists

## 1. Download to `data/raw`

In [ ]:
summary = ingest(since=SINCE, workers=WORKERS, force=FORCE)

print(f"Published on the page: {summary['discovered']}")
print(f"Selected (SINCE={SINCE}): {summary['selected']}")
print(f"New: {len(summary['new'])} | Already present: {summary['skipped']} | Failed: {len(summary['failed'])}")
if summary["failed"]:
    print("Re-run the cell to retry:", *summary["failed"], sep="\n  ")

## 2. Inventory of `data/raw/mibici`

Built from `manifest.json` (lineage: source URL, sha256, size, download date).

In [ ]:
manifest = json.loads((RAW_MIBICI_DIR / MANIFEST_NAME).read_text(encoding="utf-8"))

inventory = (
    pd.DataFrame.from_dict(manifest, orient="index")
    .rename_axis("name")
    .reset_index()
    .assign(
        kind=lambda d: d["name"].str.extract(r"^(datos_abiertos|nomenclatura)", expand=False),
        period=lambda d: d["name"].str.extract(r"(\d{4}_\d{2})", expand=False),
        mb=lambda d: (d["bytes"] / 2**20).round(1),
    )
    .sort_values("name")
)
inventory[["name", "kind", "period", "mb", "downloaded_at"]]

In [ ]:
inventory.groupby("kind").agg(files=("name", "size"), total_mb=("mb", "sum"))

## 3. Schema per file (headers only)

Reading only the header is cheap and reveals column or encoding changes between months before loading full data.

In [ ]:
def read_raw_csv(path: Path, **kwargs) -> pd.DataFrame:
    """Reads a raw CSV trying UTF-8 and then Latin-1 (MiBici is not consistent)."""
    try:
        return pd.read_csv(path, encoding="utf-8", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kwargs)


trip_files = sorted(RAW_MIBICI_DIR.glob("datos_abiertos_*.csv"))
station_files = sorted(RAW_MIBICI_DIR.glob("nomenclatura_*.csv"))

schemas = pd.DataFrame(
    {"name": p.name, "columns": tuple(read_raw_csv(p, nrows=0).columns)} for p in trip_files
)
schemas.groupby("columns")["name"].agg(["size", "first", "last"])

## 4. Trips: most recent month

In [ ]:
latest_trips = trip_files[-1]
trips = read_raw_csv(latest_trips)
print(latest_trips.name, trips.shape)
trips.head()

In [ ]:
trips.info(memory_usage="deep")

In [ ]:
print(f"Duplicated rows: {trips.duplicated().sum()}")
pd.DataFrame(
    {
        "nulls": trips.isna().sum(),
        "pct_nulls": (trips.isna().mean() * 100).round(2),
        "unique": trips.nunique(),
    }
)

In [ ]:
trips.describe(include="all").T

## 5. Stations: most recent nomenclature

In [ ]:
latest_stations = station_files[-1]
stations = read_raw_csv(latest_stations)
print(latest_stations.name, stations.shape)
print(f"Duplicated rows: {stations.duplicated().sum()}")
stations.head()

In [ ]:
stations.isna().sum().to_frame("nulls").T

## Findings and next steps

- _Record observations here (schema changes between years, encoding, nulls, duplicates)._
- Next notebook (phase 1): name normalization into `data/interim` with `team2_mlops2026.data.raw_data_normalization`, and cleaning.